# Système de recommandation agricole - Modélisation - premiers pas

### Script afin de comparer les différents modèles sur les 2 fichiers

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.svm import SVR

import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Chargement des données
df_enrichi = pd.read_csv("../data/processed/yield_df_enriched.csv")
df_conso = pd.read_csv("../data/processed/yield_df_final_conso.csv")

# Colonnes
cols_num_enrichi = [
    "avg_temp", "rainfall_mm", "pesticides_tonnes",
    "tech_trend", "irrigation_impact", "climate_instability",
    "relative_tech_intensity"
]

cols_cat_enrichi = [
    "item_Cassava", "item_Maize", "item_Plantains and others", "item_Potatoes", "item_Rice",
    "item_Sorghum", "item_Soybean", "item_Sweet potatoes", "item_Wheat",
    "item_Yams", "Weather_Condition_Cloudy", "Weather_Condition_Rainy",
    "Weather_Condition_Sunny", "Soil_Type_Chalky", "Soil_Type_Clay",
    "Soil_Type_Loam", "Soil_Type_Peaty", "Soil_Type_Sandy", "Soil_Type_Silt",
    "Agro_Profile_cold_dry_extensive", "Agro_Profile_cold_dry_intensive",
    "Agro_Profile_cold_wet_extensive", "Agro_Profile_cold_wet_intensive",
    "Agro_Profile_hot_dry_extensive", "Agro_Profile_hot_dry_intensive",
    "Agro_Profile_hot_wet_extensive", "Agro_Profile_hot_wet_intensive",
    "Fertilizer_Used_0", "Fertilizer_Used_1",
    "Irrigation_Used_0", "Irrigation_Used_1"
]

cols_num_conso = [
    "avg_temp", "rainfall_mm", "pesticides_tonnes",
    "tech_trend", "climate_instability", "relative_tech_intensity"
]

cols_cat_conso = [
    "item_Cassava", "item_Maize", "item_Plantains and others", "item_Potatoes",
    "item_Rice, paddy", "item_Sorghum", "item_Soybeans", "item_Sweet potatoes",
    "item_Wheat", "item_Yams",
    "Agro_Profile_cold_dry_extensive", "Agro_Profile_cold_dry_intensive",
    "Agro_Profile_cold_wet_extensive", "Agro_Profile_cold_wet_intensive",
    "Agro_Profile_hot_dry_extensive", "Agro_Profile_hot_dry_intensive",
    "Agro_Profile_hot_wet_extensive", "Agro_Profile_hot_wet_intensive"
]

# Fonction d'évaluation
def evaluate_dataset(
    df,
    dataset_name,
    numeric_cols,
    categorical_cols,
    model_name,
    model
):
    target_col = "yield"

    selected_cols = numeric_cols + categorical_cols + [target_col]
    data = df[selected_cols].copy()

    X = data.drop(columns=[target_col])
    y = data[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    # Preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", "passthrough", categorical_cols)
        ]
    )
    # Définition du pipeline
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    # Cross validation avec Kfold
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }
    # Enregsitrement de la cross validation
    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    # Le modèle apprend à partir des données d’entraînement
    pipeline.fit(X_train, y_train)

    # Le modèle utilise ce qu’il a appris pour prédire
    y_pred = pipeline.predict(X_test)

    cv_rmse_scores = np.sqrt(-cv_results["test_mse"])
    cv_mae_scores = -cv_results["test_mae"]
    cv_r2_scores = cv_results["test_r2"]

    # Comparer prédiction vs réalité
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_mae = mean_absolute_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

    return {
        "dataset": dataset_name,
        "model": model_name,
        "n_rows": len(data),
        "n_features": X.shape[1],
        "cv_rmse_mean": cv_rmse_scores.mean(),
        "cv_rmse_std": cv_rmse_scores.std(),
        "cv_mae_mean": cv_mae_scores.mean(),
        "cv_mae_std": cv_mae_scores.std(),
        "cv_r2_mean": cv_r2_scores.mean(),
        "cv_r2_std": cv_r2_scores.std(),
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2
    }


# Définir les modèles
models = {
    "DummyRegressor": DummyRegressor(),
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(random_state=42),
    "XGBoost": xgb.XGBRegressor(random_state=42),
    "LightGBM":lgb.LGBMRegressor(verbose=-1, random_state=42)}


# Boucler sur les datasets et les modèles
all_results = []

for model_name, model in models.items():

    result_enrichi = evaluate_dataset(
        df=df_enrichi,
        dataset_name="Fichier enrichi",
        numeric_cols=cols_num_enrichi,
        categorical_cols=cols_cat_enrichi,
        model_name=model_name,
        model=model
    )
    all_results.append(result_enrichi)

    result_conso = evaluate_dataset(
        df=df_conso,
        dataset_name="Fichier consolidé",
        numeric_cols=cols_num_conso,
        categorical_cols=cols_cat_conso,
        model_name=model_name,
        model=model
    )
    all_results.append(result_conso)


# Tableau final
comparison = pd.DataFrame(all_results)

comparison = comparison[[
    "dataset", "model",
    "n_rows", "n_features",
    "cv_rmse_mean", "cv_rmse_std",
    "cv_mae_mean", "cv_mae_std",
    "cv_r2_mean", "cv_r2_std",
    "test_rmse", "test_mae", "test_r2"
]]

pd.DataFrame(comparison.round(4).sort_values(by=["model", "test_rmse"]))

,dataset,model,n_rows,n_features,cv_rmse_mean,cv_rmse_std,cv_mae_mean,cv_mae_std,cv_r2_mean,cv_r2_std,test_rmse,test_mae,test_r2
0,Fichier enrichi,DummyRegressor,29151,38,75652.6639,1271.5658,55591.4405,612.0935,-0.0007,0.0003,73864.1004,54567.1261,-0.0012
1,Fichier consolidé,DummyRegressor,29151,24,75652.6639,1271.5658,55591.4405,612.0935,-0.0007,0.0003,73864.1004,54567.1261,-0.0012
8,Fichier enrichi,LightGBM,29151,38,28195.1980,830.2414,17920.1400,314.8971,0.8609,0.0064,27703.8593,17454.4365,0.8592
9,Fichier consolidé,LightGBM,29151,24,27954.8256,876.4924,17768.2938,310.4446,0.8633,0.0071,27956.1463,17625.7264,0.8566
2,Fichier enrichi,LinearRegression,29151,38,51310.2734,1258.0180,33432.7746,650.7589,0.5397,0.0096,50982.7735,33071.1113,0.5230
3,Fichier consolidé,LinearRegression,29151,24,51341.9533,1271.3610,33444.5241,675.6013,0.5392,0.0096,51047.0116,33133.5267,0.5218
4,Fichier enrichi,RandomForest,29151,38,20799.5607,772.1689,10226.1873,294.6239,0.9243,0.0051,19681.5670,9289.9715,0.9289
5,Fichier consolidé,RandomForest,29151,24,20383.3497,855.5534,9515.7169,295.8420,0.9272,0.0056,19686.6864,8720.9609,0.9289
7,Fichier consolidé,XGBoost,29151,24,22671.0428,684.7109,13549.7758,299.5553,0.9101,0.0046,22180.2492,12803.4932,0.9097
6,Fichier enrichi,XGBoost,29151,38,23464.4425,536.1863,14145.8625,277.0853,0.9037,0.0033,22706.2423,13230.0410,0.9054


- Sur ces premiers tests on ne voit pas des meilleurs résultats pour le fichier enrichi, les informations rajoutées ne semblent pas apporter quelque chose d'utile pour nos modèles.
- On voit que le modèle de RandomForest a des meilleurs résulats (sans optimisation des meilleurs paramètres) mais suivi de très près par XGBoost et LightGBM.
- N'ayant pas une distribution linéaire, on voit que la regression linéraire ne généralise pas.